# Step 1 : Environment Setup and Library Imports

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
dataset_base_path = "./drive/MyDrive/Brain_Tumor_Classification/dataset_extracted/Brain-Tumor-Classification-DataSet-master"
train_dir = os.path.join(dataset_base_path, 'Training')
val_dir = os.path.join(dataset_base_path, 'Testing')

print(" Environment successfully prepared for Upgrade.")

# Step 2 : Advanced Medical Data Augmentation (RandomAffine & RandomErasing)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    # Added RandomAffine for elastic-like brain tissue deformations
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # Added RandomErasing to force the model to look at alternative tumor features
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3))
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
val_dataset = ImageFolder(root=val_dir, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Datasets loaded successfully. Classes: {train_dataset.classes}")
print(f"Advanced Augmentations Active! Ready to feed the new architecture.")

# Step 3 : Upgrading Backbone to EfficientNet_B0

In [ ]:
# EfficientNet handles fine-grained feature scaling much better than ResNet
model = models.efficientnet_b0(pretrained=True)

# Replace the classifier layer to match our 4 tumor classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Unfreeze ALL layers for comprehensive deep fine-tuning
for param in model.parameters():
    param.requires_grad = True

print(f" EfficientNet_B0 successfully initialized on {device.type.upper()}. All parameters unfrozen.")

# Step 4 : Dynamic Loss and Anti-Overfitting Optimizer Configuration



In [ ]:
# Keep the 2.5 weight on Class 0 (Glioma) to enforce precision/recall balancing
weights = torch.tensor([2.5, 1.0, 1.0, 1.0], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# Micro learning rate for a smoother fine-tuning of the advanced backbone
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0002)

print("High-precision Optimizer ready.")

# Step 5 : Smart Training Loop with Validation-Driven Checkpointing

In [ ]:
epochs = 15
start_epoch = 0
best_val_acc = 0.0

checkpoint_dir = "./drive/MyDrive/Brain_Tumor_Classification"
checkpoint_path_day7 = os.path.join(checkpoint_dir, 'best_brain_tumor_model_day7.pth')

# ==AUTO-RESUME==
if os.path.exists(checkpoint_path_day7):
    print(" Found existing efficientnet checkpoint! Loading weights to resume...")
    checkpoint = torch.load(checkpoint_path_day7, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch']
    best_val_acc = checkpoint['accuracy']
    print(f" Resuming successfully from Epoch {start_epoch + 1} with previous Best Val Acc: {best_val_acc:.2f}%")
else:
    print(" No existing checkpoint found. Starting training from scratch.")
# =================================================================

scaler = torch.amp.GradScaler('cuda')

for epoch in range(start_epoch, epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train, total_train = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = (correct_train / total_train) * 100

    # --- VALIDATION / TESTING PHASE ---
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = (val_correct / val_total) * 100

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% || Val Acc: {val_acc:.2f}%")


    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': val_acc
        }, checkpoint_path_day7)
        print(f"--> 🔥 New Record! Saved Checkpoint with Val Acc: {val_acc:.2f}%")

    torch.cuda.empty_cache()

print(f"\n Efficientnet training Complete! Best verifiable Validation Accuracy: {best_val_acc:.2f}%")

⏳ Found existing Day 7 checkpoint! Loading weights to resume...
▶️ Resuming successfully from Epoch 8 with previous Best Val Acc: 78.43%
🚀 Launching Day 7 Architecture Optimization...
Epoch [8/15] | Train Loss: 0.0802 | Train Acc: 96.76% || Val Acc: 76.90%
Epoch [9/15] | Train Loss: 0.0647 | Train Acc: 97.80% || Val Acc: 81.98%
--> 🔥 New Record! Saved Day 7 Checkpoint with Val Acc: 81.98%
Epoch [10/15] | Train Loss: 0.0517 | Train Acc: 98.12% || Val Acc: 79.44%
Epoch [11/15] | Train Loss: 0.0480 | Train Acc: 98.22% || Val Acc: 76.40%
Epoch [12/15] | Train Loss: 0.0626 | Train Acc: 97.91% || Val Acc: 79.95%
Epoch [13/15] | Train Loss: 0.0479 | Train Acc: 98.40% || Val Acc: 74.87%


# Step 6 : Final Evaluation and Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

if os.path.exists(checkpoint_path_day7):
    checkpoint = torch.load(checkpoint_path_day7, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Successfully loaded Day 7 best weights (Val Acc: {checkpoint['accuracy']:.2f}%)")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=val_dataset.classes, yticklabels=val_dataset.classes)
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix - Day 7 EfficientNet Upgrade')
plt.show()

print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))